# Baire Class 1 Functions as Pointwise Limits
## From Continuous Approximations to the Tame/Wild Boundary

**Companion notebook** — *Complexity of Deep Computations via Topology* [Dueñez, Iovino, Matos-Wiederhold, Salvetti, Tall]

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/public-site/blob/main/2026-FES_Acatlan-ultracomputos/notebooks/baire_functions.ipynb)

---

A deep computation is a sequence of states
$$z_0 \;\xrightarrow{f}\; z_1 \;\xrightarrow{f}\; z_2 \;\xrightarrow{f}\; \cdots$$
and its **deep equilibrium** is the pointwise limit $z_\infty = \lim_{n\to\infty} f^{\circ n}(z_0)$
— a function of the starting point $z_0$.

Whether this limit function is *well-behaved* or *wild* is measured by the **Baire class hierarchy**:

| Class | Definition | Example |
|-------|-----------|---------|
| Baire-0 | Continuous | $\sin(x)$, $e^x$, polynomials |
| **Baire-1** | Pointwise limit of continuous functions | Thomae's function, $\text{sgn}(x)$ |
| Baire-2 | Pointwise limit of Baire-1 functions | Dirichlet's function $\mathbf{1}_{\mathbb{Q}}$ |

The **BFT theorem** (Bourgain–Fremlin–Talagrand) identifies Baire-1 precisely as
the class of *tame* deep computations: those satisfying the NIP.
This notebook builds that identification from the ground up, through three
progressively subtle examples.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from fractions import Fraction
%matplotlib inline


In [ ]:
# ── Thomae's function ─────────────────────────────────────────────────────────

def thomae_scatter(N, x_min=0.0, x_max=1.0):
    # Return (xs, ys) for all rationals p/q in [x_min, x_max] with q <= N.
    xs, ys = [], []
    for q in range(1, N + 1):
        p_lo = math.ceil(x_min * q)
        p_hi = math.floor(x_max * q)
        for p in range(p_lo, p_hi + 1):
            if math.gcd(abs(p), q) == 1:
                xs.append(p / q)
                ys.append(1.0 / q)
    # Add x=0 (convention T(0)=1) and x=1 manually
    if x_min == 0 and (0, 1) not in zip(xs, ys):
        xs.append(0.0)
        ys.append(1.0)
    return np.array(xs), np.array(ys)


def thomae_approx_continuous(x_arr, N, x_min=0.0, x_max=1.0, bump_fraction=0.35):
    # Continuous approximation T_N: piecewise-linear bumps at each rational p/q, q<=N.
    # bump_fraction controls the width relative to 1/N^2 (the typical gap).
    result = np.zeros_like(x_arr, dtype=float)
    min_gap = 1.0 / (N * N)
    half_w  = bump_fraction * min_gap / 2.0
    for q in range(1, N + 1):
        p_lo = math.ceil(x_min * q)
        p_hi = math.floor(x_max * q)
        for p in range(p_lo, p_hi + 1):
            if math.gcd(abs(p), q) == 1:
                center = p / q
                height = 1.0 / q
                tent   = np.maximum(0.0,
                             height * (1.0 - np.abs(x_arr - center) / half_w))
                result = np.maximum(result, tent)
    return result


# ── Double-limit family  f_{n,m}(x) = cos^{2n}(m! * pi * x) ─────────────────

def f_nm(x_arr, n, m):
    # cos^(2n)(m! * pi * x)  via fractional-part trick to avoid overflow.
    mfact = math.factorial(m)
    frac  = (mfact * x_arr) % 1.0        # fractional part of m! * x
    cos_sq = np.cos(np.pi * frac) ** 2   # = cos^2(pi * {m! x})
    # Clip before raising to power to suppress underflow warnings
    return np.where(cos_sq > 1e-300, cos_sq ** n, 0.0)


def g_m_spikes(m, x_min=0.0, x_max=1.0):
    # Return x-locations where g_m = lim_{n->inf} f_{n,m} equals 1.
    mfact = math.factorial(m)
    k_lo  = math.ceil(x_min * mfact)
    k_hi  = math.floor(x_max * mfact)
    return np.array([k / mfact for k in range(k_lo, k_hi + 1)])


## Example 1 — $f_n(x) = x^n$ : the simplest Baire-1 story

The sequence $f_n(x) = x^n$ on $[0, 1]$ consists of continuous functions.
Their pointwise limit is
$$f(x) = \lim_{n\to\infty} x^n = \begin{cases} 0 & 0 \leq x < 1 \\ 1 & x = 1. \end{cases}$$

- $f$ is **Baire class 1**: a pointwise limit of continuous $f_n$.
- $f$ is **discontinuous** at $x = 1$ — the convergence is *not uniform* near $x=1$.
- Yet $f$ is continuous **almost everywhere** (except at a single point, a meager set).

**Baire's theorem**: A Baire-1 function on a Polish space is continuous on a
*comeager* (topologically large) set — the set of discontinuities is
*meager* (topologically negligible, a countable union of nowhere-dense sets).

Use the slider to watch $f_n$ collapse to $f$.


In [ ]:
@widgets.interact(
    n=widgets.IntSlider(
        value=1, min=1, max=150, step=1,
        description='n',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
    ),
    zoom_corner=widgets.Checkbox(value=False,
        description='Zoom into x = 1 corner'),
)
def xn_plot(n, zoom_corner):
    x = np.linspace(0.0, 1.0, 2000)
    fn = x ** n

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, xlim in zip(axes, [(0, 1), (1 - 3.0/n**0.6, 1)] if zoom_corner else [(0,1),(0,1)]):
        ax.plot(x, fn, color='steelblue', lw=2, label=f'$f_n(x)=x^n$, $n={n}$')
        ax.axhline(0, color='gray', lw=1, ls='--')
        # Limit function: 0 on [0,1), 1 at x=1
        ax.plot([0, 1 - 1e-4], [0, 0], color='firebrick', lw=2.5,
                label=r'Limit $f(x)$')
        ax.plot([1], [1], 'o', color='firebrick', ms=9, zorder=5)
        ax.plot([1], [0], 'o', color='firebrick', ms=9, mfc='white',
                markeredgecolor='firebrick', zorder=5)

        # Highlight the "transition zone" width ~1/sqrt(n)
        transition_width = 1.0 / n**0.5
        ax.axvspan(max(0, 1 - transition_width), 1, alpha=0.12, color='gold',
                   label=f'Transition zone width $\\sim 1/\\sqrt{{n}}$')

        if zoom_corner:
            ax.set_xlim(max(0, 1 - 3.0/n**0.6), 1.02)
        ax.set_ylim(-0.05, 1.12)
        ax.set_xlabel('$x$', fontsize=13)
        ax.set_ylabel('$f_n(x)$', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.25)

    axes[0].set_title(
        f'$f_n(x) = x^n \\to f(x)$ pointwise  ($n = {n}$)\n'
        'Baire-1: continuous approximations collapse to a discontinuous limit',
        fontsize=11,
    )
    axes[1].set_title(
        'Zoomed: convergence near the discontinuity at $x=1$' if zoom_corner
        else 'Same view (check "Zoom" to magnify near $x=1$)',
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()


## Example 2 — Thomae's function (the "ruler function")

**Thomae's function** (also called the *ruler function* or *popcorn function*) is
$$T(x) = \begin{cases}
  1/q & \text{if } x = p/q \text{ in lowest terms, } q > 0 \\
  0   & \text{if } x \text{ is irrational.}
\end{cases}$$

The name "ruler function" is apt: the values $1/q$ at rationals of denominator $q$
look exactly like the markings on a ruler — tallest at $1/2$, shorter at $1/3$,
$1/4$, etc., down to imperceptible heights at large denominators.

**$T$ is Baire class 1.** Construct a continuous approximation:
for each $N$, let $T_N(x)$ be the function that spikes to height $1/q$ near
each rational $p/q$ with $q \leq N$, and is zero elsewhere.
Then $T_N$ is continuous (piecewise linear bumps) and $T_N \to T$ pointwise.

**$T$ is continuous at every irrational.** Near any irrational $x_0$, all rationals
$p/q$ with $q \leq N$ are bounded away from $x_0$, so $T(x) \leq 1/N$ for $x$
near $x_0$ — and $1/N \to 0$.
Discontinuities occur only at rationals, which form a *meager* (countable) set.
This confirms Baire's theorem: the discontinuity set of a Baire-1 function is meager.


In [ ]:
@widgets.interact(
    N=widgets.IntSlider(
        value=5, min=1, max=30, step=1,
        description='Max denominator N',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
    ),
    show_approx=widgets.Checkbox(value=True,
        description='Show continuous approximation $T_N$'),
)
def thomae_plot(N, show_approx):
    xs_sc, ys_sc = thomae_scatter(N)
    x_fine = np.linspace(0.0, 1.0, 6000)

    n_rationals = len(xs_sc)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: scatter of Thomae values at rationals p/q, q <= N ──────────────
    ax = axes[0]
    ax.scatter(xs_sc, ys_sc, s=10, c=ys_sc, cmap='plasma_r', vmin=0, vmax=1,
               zorder=4, label=f'$T(p/q)=1/q$,  $q \\leq {N}$')
    ax.axhline(0, color='gray', lw=1, ls='--', label='$T(x)=0$ at irrationals')

    # Mark denominator levels
    for q in range(1, min(N+1, 8)):
        ax.axhline(1/q, color='lightgray', lw=0.6, ls=':')
        ax.text(1.01, 1/q, f'$1/{q}$', va='center', fontsize=8, color='gray')

    ax.set_xlim(-0.02, 1.08)
    ax.set_ylim(-0.03, 1.12)
    ax.set_xlabel('$x$', fontsize=13)
    ax.set_ylabel('$T(x)$', fontsize=13)
    ax.set_title(
        f"Thomae's function: {n_rationals} rational points with $q \\leq {N}$\n"
        '"Ruler function": height $1/q$ at $p/q$, zero at irrationals',
        fontsize=11,
    )
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.2)

    # ── Right: continuous approximation T_N ───────────────────────────────────
    ax2 = axes[1]
    if show_approx:
        t_approx = thomae_approx_continuous(x_fine, N)
        ax2.plot(x_fine, t_approx, color='steelblue', lw=1.2,
                 label=f'$T_{{{N}}}(x)$ (continuous approximation)')
    ax2.scatter(xs_sc, ys_sc, s=8, color='firebrick', zorder=5,
                label=f'$T(p/q)$,  $q \\leq {N}$ (exact Thomae)')
    ax2.axhline(0, color='gray', lw=1, ls='--', alpha=0.6)

    ax2.set_xlim(-0.02, 1.02)
    ax2.set_ylim(-0.03, 1.12)
    ax2.set_xlabel('$x$', fontsize=13)
    ax2.set_ylabel('$T_N(x)$', fontsize=13)
    ax2.set_title(
        f'Continuous approximation $T_{{{N}}} \\to T$ as $N \\to \\infty$\n'
        'Bumps narrow and sink; in the limit: Baire-1 function',
        fontsize=11,
    )
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


## Example 3 — Dirichlet's function: Baire-2 but *not* Baire-1

**Dirichlet's function** is the indicator of the rationals:
$$D(x) = \mathbf{1}_{\mathbb{Q}}(x) = \begin{cases} 1 & x \in \mathbb{Q} \\ 0 & x \notin \mathbb{Q}. \end{cases}$$

**$D$ is nowhere continuous** — at every point, every neighbourhood contains both
rationals (where $D=1$) and irrationals (where $D=0$).
By Baire's theorem, this means $D$ **cannot be Baire class 1**.

**$D$ is Baire class 2.** It arises as a *double limit*:
$$D(x) = \lim_{m\to\infty} \lim_{n\to\infty} \cos^{2n}(m!\,\pi x).$$

Here is what the double limit does:
- **Inner limit** (fix $m$, let $n\to\infty$): $g_m(x) = \lim_{n\to\infty}\cos^{2n}(m!\pi x)$
  equals $1$ when $m!\,x \in \mathbb{Z}$ (i.e., $x$ is a rational $k/(m!)$)
  and $0$ otherwise. Each $g_m$ is Baire-1 (a limit of continuous $\cos^{2n}$).
- **Outer limit** (let $m\to\infty$): as $m$ grows, $m!$ is divisible by all denominators
  up to $m$, so $g_m(x) \to 1$ for every rational $x$ — recovering $D$.

The interactive plot below shows the intermediate functions $f_{n,m}$ and $g_m$
for small $m$, where the spike structure is visible at normal resolution.


In [ ]:
@widgets.interact(
    m=widgets.IntSlider(
        value=2, min=1, max=6, step=1,
        description='Outer index m',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
    n=widgets.IntSlider(
        value=10, min=1, max=200, step=1,
        description='Inner index n',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='460px'),
    ),
)
def dirichlet_plot(m, n):
    mfact = math.factorial(m)
    x_fine = np.linspace(0.0, 1.0, max(8000, mfact * 20))
    fn_vals = f_nm(x_fine, n, m)
    spike_xs = g_m_spikes(m)

    n_spikes = len(spike_xs)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: f_{n,m}(x) — the continuous approximation ──────────────────────
    ax = axes[0]
    ax.plot(x_fine, fn_vals, color='steelblue', lw=1.0,
            label=f'$f_{{n,m}}(x)=\\cos^{{2n}}(m!\\pi x)$,  $n={n}$, $m={m}$')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.axhline(1, color='firebrick', lw=0.8, ls='--', alpha=0.5)

    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.05, 1.35)
    ax.set_xlabel('$x$', fontsize=13)
    ax.set_ylabel('$f_{n,m}(x)$', fontsize=13)
    ax.set_title(
        f'$f_{{n,m}}(x) = \\cos^{{2n}}(m!\\pi x)$  for  $m={m}$, $n={n}$\n'
        f'$m! = {mfact}$: continuous, with {n_spikes} spikes approaching height 1',
        fontsize=11,
    )
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # ── Right: inner limit g_m = lim_{n->inf} f_{n,m} ────────────────────────
    ax2 = axes[1]
    # Show spikes at height 1 (the inner limit)
    ax2.scatter(spike_xs, np.ones(n_spikes), s=15, color='steelblue',
                zorder=5, label=f'$g_m(x)=1$ at $x = k/{mfact}$  ({n_spikes} pts)')
    ax2.scatter(spike_xs, np.zeros(n_spikes), s=4, color='gray', alpha=0.3, zorder=3)
    ax2.axhline(0, color='gray', lw=1, ls='--',
                label='$g_m(x)=0$ elsewhere (irrationals and other rationals)')

    # Overlay: Dirichlet = outer limit (schematic: rational = orange, irrational = gray 0)
    from matplotlib.patches import Patch
    dirichlet_patch = Patch(color='firebrick', alpha=0.6,
                            label='Dirichlet $D(x)$: limit as $m\\to\\infty$'
                                  '(1 at ALL rationals, 0 at irrationals)')
    ax2.axhline(1, color='firebrick', lw=1.5, ls=':', alpha=0.5)

    ax2.set_xlim(-0.01, 1.01)
    ax2.set_ylim(-0.05, 1.35)
    ax2.set_xlabel('$x$', fontsize=13)
    ax2.set_ylabel('value', fontsize=13)
    ax2.set_title(
        f'Inner limit $g_m = \\lim_{{n\\to\\infty}} f_{{n,m}}$  for  $m={m}$\n'
        f'Baire-1 function: spikes at $k/{mfact}$, then $m\\to\\infty$ gives Dirichlet',
        fontsize=11,
    )
    ax2.legend(handles=[ax2.get_legend_handles_labels()[0][0],
                         ax2.get_legend_handles_labels()[0][1],
                         dirichlet_patch],
               fontsize=9, loc='upper right')
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


## Limit exchange failure and the Baire class boundary

The double limit $\lim_{m\to\infty}\lim_{n\to\infty} f_{n,m}(x) = D(x)$
is a Baire-2 function.

What happens if we **exchange the order of limits**?
$$\lim_{n\to\infty} \lim_{m\to\infty} f_{n,m}(x)
  = \lim_{n\to\infty} \lim_{m\to\infty} \cos^{2n}(m!\,\pi x).$$
For irrational $x$: the sequence $m!\,\pi x \bmod \pi$ is equidistributed,
so $\lim_{m\to\infty} |\cos(m!\pi x)| < 1$ in an averaged sense,
and $\cos^{2n}(m!\pi x)^{1/n} \to 0$; thus the inner limit is 0.
For rational $x$: eventually $m!x \in \mathbb{Z}$, so the inner limit is 1.
In either case the outer limit collapses to $D(x)$ again — but the order matters
in a deeper sense captured by the **Grothendieck criterion**:

> *A bounded sequence of functions $(f_n)$ on a compact space $K$ is relatively
> compact in the Baire-1 topology (i.e., has a Baire-1 cluster point) if and only if
> every iterated limit $\lim_m \lim_n f_n(x_m)$ equals $\lim_n \lim_m f_n(x_m)$
> for every sequence $(x_m)$ in $K$.*

**This is the limit-exchange criterion** from the research paper.
Baire-1 functions (tame deep computations) are exactly those for which the order
of iterated limits can always be exchanged.
Dirichlet's function $D$ *fails* this criterion — it is wild.

The plot below makes this concrete for the sequence $g_m(x) = \mathbf{1}_{x = k/(m!)}$
and a judicious sequence $(x_m)$ of rationals converging to an irrational.


In [ ]:
# Limit exchange failure: visualize lim_m lim_n f_{n,m}(x) vs.
# what a rational / irrational witnesses.
#
# Strategy: pick a sequence x_m of rationals converging to an irrational x_inf.
# At each stage m, g_m(x_m) = 1 (since x_m is rational with denom <= m!).
# So  lim_m g_m(x_m) = lim_m 1 = 1.
# But g_m(x_inf) = 0 for all m (x_inf is irrational, never equals k/(m!)).
# So  lim_m g_m(x_inf) = 0.
# Therefore:  lim_m [ lim_n f_{n,m}(x_m) ] = 1   (approach along rationals)
#             lim_m [ lim_n f_{n,m}(x_inf)] = 0   (target is irrational)
# This is the independence pattern that makes Dirichlet IP (not NIP).

x_inf = 1.0 / math.sqrt(2)   # irrational target  (1/sqrt(2) = 0.70710...)
M_MAX = 6

# x_m = nearest spike of g_m to x_inf, i.e. round(x_inf * m!) / m!
# By construction g_m(x_m) = 1 (x_m IS a spike) and g_m(x_inf) = 0 (irrational).
x_ms   = [round(x_inf * math.factorial(m)) / math.factorial(m) for m in range(1, M_MAX+1)]
x_labs = [f'{round(x_inf*math.factorial(m))}/{math.factorial(m)}' for m in range(1, M_MAX+1)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: g_m(x_inf) and g_m(x_m) for m = 1 ... M_MAX ──────────────────────
ms = np.arange(1, M_MAX + 1)
g_at_irrational = np.zeros(M_MAX)   # g_m(x_inf) = 0 for all m
g_at_rational   = np.ones(M_MAX)    # g_m(x_m) = 1 by construction

ax = axes[0]
ax.plot(ms, g_at_irrational, 's-', color='steelblue', lw=2, ms=10,
        label=f'$g_m(x_\\infty)=0$  [$x_\\infty = 1/\\sqrt{{2}}$, irrational]')
ax.plot(ms, g_at_rational, 'o-', color='firebrick', lw=2, ms=10,
        label='$g_m(x_m)=1$  [$x_m = $ nearest spike of $g_m$ to $x_\\infty$]')

for m, lbl in zip(ms, x_labs):
    ax.annotate(lbl, xy=(m, 1.03), ha='center', fontsize=8, color='firebrick')

ax.set_xlim(0.5, M_MAX + 0.5)
ax.set_ylim(-0.15, 1.25)
ax.set_xticks(ms)
ax.set_xlabel('Outer index $m$', fontsize=12)
ax.set_ylabel('$g_m(x)$', fontsize=12)
ax.set_title(
    'Limit exchange failure\n'
    '$\\lim_m\\,g_m(x_m)=1 \\neq 0 = \\lim_m\\,g_m(x_\\infty)$',
    fontsize=12,
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Right: independence matrix ────────────────────────────────────────────────
ax2 = axes[1]
# Columns: x_inf (irrational) then x_1,...,x_{M_MAX} (rational spikes)
# Rows: m = 1,...,M_MAX
# Entry: f_{200,m}(x) approximates g_m(x) in {0,1}
labels_x  = ['$x_{\\infty}\\approx 0.7071$'] + [f'$x_{m}$={lbl}' for m, lbl in zip(ms, x_labs)]
all_xs    = [x_inf] + x_ms
matrix    = np.array([[f_nm(np.array([x]), 200, m)[0] for x in all_xs]
                       for m in ms])

im = ax2.imshow(matrix, aspect='auto', cmap='RdYlGn',
                vmin=0, vmax=1, origin='lower')
ax2.set_xticks(range(len(all_xs)))
ax2.set_xticklabels(labels_x, fontsize=9)
ax2.set_yticks(range(M_MAX))
ax2.set_yticklabels([f'm={m}' for m in ms], fontsize=10)
plt.colorbar(im, ax=ax2, label='$f_{200,m}(x)$  (approximates $g_m(x)$)')
ax2.set_title(
    'Independence pattern (IP): the 0/1 matrix\n'
    'Green=1 (spike), Red=0 (no spike) — columns are never all the same',
    fontsize=11,
)

plt.tight_layout()
plt.show()

print('The right panel shows the IP: for any m, the rational x_m gives 1')
print('but x_inf gives 0. This infinite 0/1 pattern is the signature of Baire-2.')
print()
print('A Baire-1 (NIP) function could NOT produce such an independent pattern.')


## The full picture: Baire class, NIP, and deep computations

| Concept | Analytical | Combinatorial | Computational |
|---------|-----------|---------------|---------------|
| Baire class 0 | Continuous | — | Finite depth |
| **Baire class 1** | Pointwise limit of cts. | **NIP** (finite VC dim.) | **Tame** deep equilibrium |
| Baire class 2 | Limit of Baire-1 | IP (infinite VC dim.) | **Wild**: limit not useful |

**The BFT theorem** closes the loop:

> *For a pointwise bounded sequence of continuous functions on a Polish space,
> the following are equivalent:
> (i) relative compactness in the Baire-1 topology;
> (ii) every cluster point is Baire class 1;
> (iii) the sequence satisfies the NIP (no infinite independent pattern);
> (iv) the limit-exchange criterion holds.*

**In the deep-computation framework:**
- Thomae's function = prototype of a tame deep equilibrium (Baire-1, NIP).
  Its discontinuity set (the rationals) is meager — the "Julia set" of this computation
  is negligible in the topological sense.
- Dirichlet's function = prototype of a wild deep computation (Baire-2, IP).
  Its discontinuity set is the *whole domain* — no stable basin of attraction exists.

The Newton fractal notebook made this dichotomy visual in the complex plane.
This notebook makes it precise on the real line.
